# EGIT 2024: preparación para clustering
Consolidación de los cuatro trimestres, con una fila por persona y trimestre como unidad provisional. No se asume seguimiento de la misma persona entre trimestres.

Estructura adaptada de los puntos 0–4 del parcial. Exploración, faltantes e IQR: `funciones_notebooks_profesor.txt`. Cruces, duplicados y reorganización de índices: código del parcial. Los ciclos recorren archivos o columnas, no personas. La exportación únicamente guarda resultados.

Las interpretaciones y decisiones son del estudiante. Listas vacías significan **pendiente**. No se ejecutan PCA ni k-means.

## Punto 0: Carga de datos
Se leen los 36 CSV extraídos, separados por `|`, sin cargar nuevamente los TXT ni los ZIP. Se conservan códigos como texto y ceros iniciales: tener dígitos no convierte una variable en cuantitativa.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

RAIZ = Path.cwd()
assert (RAIZ / "Bases de datos").is_dir(), "Ejecuta desde K_means."
SALIDA = RAIZ / "Resultados"
SALIDA.mkdir(exist_ok=True)
carpetas = {
    1: "Base de datos-EGIT-Base Anonimizada-I trim2024",
    2: "Base de datos-EGIT-Base Anonimizada-II trim2024",
    3: "III Trim 2024",
    4: "Base de datos-EGIT-Base Anonimizada-IV trim2024",
}
modulos = ["CARACTERISTICAS_GENERALES", "EDUCACION", "FUERZA_DE_TRABAJO",
           "COMPLEMENTARIAS", "TURISMO", "EXCURSIONISMO", "HOGAR",
           "VIVIENDA", "VIVIENDA_USO_RECREATIVO"]
tablas = {}
inventario = []
for trimestre, carpeta in carpetas.items():
    for modulo in modulos:
        ruta = RAIZ / "Bases de datos" / carpeta / (modulo + ".csv")
        tabla = pd.read_csv(ruta, sep="|", dtype="string", encoding="utf-8",
                            keep_default_na=False, na_values=[""])
        assert tabla.columns.is_unique
        tablas[trimestre, modulo] = tabla
        inventario.append({
            "trimestre": trimestre, "modulo": modulo,
            "filas": len(tabla), "columnas": tabla.shape[1],
            "duplicados_exactos": int(tabla.duplicated().sum()),
            "archivo": str(ruta.relative_to(RAIZ)),
        })
inventario = pd.DataFrame(inventario)
assert len(inventario) == 36
display(inventario)

,trimestre,modulo,filas,columnas,duplicados_exactos,archivo
0,1,CARACTERISTICAS_GENERALES,31185,12,0,Bases de datos\Base de datos-EGIT-Base Anonimi...
1,1,EDUCACION,30291,12,0,Bases de datos\Base de datos-EGIT-Base Anonimi...
2,1,FUERZA_DE_TRABAJO,27401,17,0,Bases de datos\Base de datos-EGIT-Base Anonimi...
3,1,COMPLEMENTARIAS,27401,31,0,Bases de datos\Base de datos-EGIT-Base Anonimi...
4,1,TURISMO,27401,92,0,Bases de datos\Base de datos-EGIT-Base Anonimi...
5,1,EXCURSIONISMO,27401,49,0,Bases de datos\Base de datos-EGIT-Base Anonimi...
6,1,HOGAR,10746,23,0,Bases de datos\Base de datos-EGIT-Base Anonimi...
7,1,VIVIENDA,10635,13,0,Bases de datos\Base de datos-EGIT-Base Anonimi...
8,1,VIVIENDA_USO_RECREATIVO,171,13,0,Bases de datos\Base de datos-EGIT-Base Anonimi...
9,2,CARACTERISTICAS_GENERALES,30542,12,0,Bases de datos\Base de datos-EGIT-Base Anonimi...


## Punto 1: Entendimiento y exploración inicial
### 1.1 Dimensiones, tipos, faltantes y esquemas
Son conteos de registros, no estimaciones de población ponderadas por `Fex_c`.

In [2]:
calidad_origen = []
esquemas = []
for (trimestre, modulo), tabla in tablas.items():
    calidad_origen.append(pd.DataFrame({
        "trimestre": trimestre, "modulo": modulo, "variable": tabla.columns,
        "tipo_lectura": tabla.dtypes.astype(str).values,
        "faltantes": tabla.isna().sum().values,
        "porcentaje_faltante": (tabla.isna().mean() * 100).values,
        "valores_distintos": tabla.nunique().values,
    }))
    referencia = tablas[1, modulo].columns
    esquemas.append({
        "trimestre": trimestre, "modulo": modulo,
        "mismo_esquema_que_T1": tabla.columns.equals(referencia),
        "columnas_nuevas": ", ".join(tabla.columns.difference(referencia)),
        "columnas_ausentes": ", ".join(referencia.difference(tabla.columns)),
    })
calidad_origen = pd.concat(calidad_origen, ignore_index=True)
esquemas = pd.DataFrame(esquemas)
display(esquemas)
display(calidad_origen.sort_values("porcentaje_faltante", ascending=False).head(25))

,trimestre,modulo,mismo_esquema_que_T1,columnas_nuevas,columnas_ausentes
0,1,CARACTERISTICAS_GENERALES,True,,
1,1,EDUCACION,True,,
2,1,FUERZA_DE_TRABAJO,True,,
3,1,COMPLEMENTARIAS,True,,
4,1,TURISMO,True,,
5,1,EXCURSIONISMO,True,,
6,1,HOGAR,True,,
7,1,VIVIENDA,True,,
8,1,VIVIENDA_USO_RECREATIVO,True,,
9,2,CARACTERISTICAS_GENERALES,True,,


,trimestre,modulo,variable,tipo_lectura,faltantes,porcentaje_faltante,valores_distintos
736,3,EXCURSIONISMO,P7587S8A3,string,26962,100.000000,0
721,3,EXCURSIONISMO,P7587S4A3,string,26962,100.000000,0
998,4,EXCURSIONISMO,P7587S8A3,string,26017,100.000000,0
474,2,EXCURSIONISMO,P7587S8A3,string,26869,100.000000,0
463,2,EXCURSIONISMO,P7587S5A3,string,26869,100.000000,0
672,3,TURISMO,P7581S5A3,string,26962,100.000000,0
422,2,TURISMO,P7581S8A3,string,26869,100.000000,0
471,2,EXCURSIONISMO,P7587S7A3,string,26869,100.000000,0
459,2,EXCURSIONISMO,P7587S4A3,string,26869,100.000000,0
424,2,TURISMO,P7581S11A2,string,26869,100.000000,0


### 1.3 Interpretación del estudiante
Pendiente: qué representa cada fila, población de interés y variables que podrían describir los grupos.

Los códigos `P...` requieren consultar preguntas, categorías y saltos del [diccionario oficial EGIT 2024](https://microdatos.dane.gov.co/index.php/catalog/864/data-dictionary).

## Punto 2: Calidad de datos y construcción de la base
### 2.0 Estado inicial: granularidad y duplicados
Módulos personales: `Directorio + Secuencia_p + Orden`. Hogar: `llave_hogar`. Vivienda: `Directorio`. Se añade el trimestre a la llave final.

En vivienda recreativa, se conserva cada registro dentro del hogar en columnas separadas. Su `Orden` no se cruza con el orden de una persona. No se suman ni promedian estos registros.

In [3]:
llave_persona = ["Directorio", "Secuencia_p", "Orden"]
llaves = {modulo: llave_persona for modulo in modulos}
llaves["HOGAR"] = ["llave_hogar"]
llaves["VIVIENDA"] = ["Directorio"]
llaves["VIVIENDA_USO_RECREATIVO"] = ["llave_hogar", "Orden"]
control_llaves = []
for (trimestre, modulo), tabla in tablas.items():
    clave = llaves[modulo]
    control_llaves.append({
        "trimestre": trimestre, "modulo": modulo, "llave": " + ".join(clave),
        "filas_sin_llave": int(tabla[clave].isna().any(axis=1).sum()),
        "duplicados_por_llave": int(tabla.duplicated(clave).sum()),
        "duplicados_exactos": int(tabla.duplicated().sum()),
    })
control_llaves = pd.DataFrame(control_llaves)
display(control_llaves)
assert control_llaves["filas_sin_llave"].sum() == 0
assert control_llaves["duplicados_por_llave"].sum() == 0, "Revisar llaves repetidas antes de cruzar."

,trimestre,modulo,llave,filas_sin_llave,duplicados_por_llave,duplicados_exactos
0,1,CARACTERISTICAS_GENERALES,Directorio + Secuencia_p + Orden,0,0,0
1,1,EDUCACION,Directorio + Secuencia_p + Orden,0,0,0
2,1,FUERZA_DE_TRABAJO,Directorio + Secuencia_p + Orden,0,0,0
3,1,COMPLEMENTARIAS,Directorio + Secuencia_p + Orden,0,0,0
4,1,TURISMO,Directorio + Secuencia_p + Orden,0,0,0
5,1,EXCURSIONISMO,Directorio + Secuencia_p + Orden,0,0,0
6,1,HOGAR,llave_hogar,0,0,0
7,1,VIVIENDA,Directorio,0,0,0
8,1,VIVIENDA_USO_RECREATIVO,llave_hogar + Orden,0,0,0
9,2,CARACTERISTICAS_GENERALES,Directorio + Secuencia_p + Orden,0,0,0


### 2.1 Ensamblaje por trimestre
Se conserva a todas las personas de características generales. Los prefijos indican procedencia y evitan sobrescribir columnas. Los indicadores `presente__MODULO` distinguen la ausencia de un registro completo de un vacío dentro del registro; **no certifican que una pregunta no aplique**.

Cada cruce comprueba unicidad, registros sin enlace y conservación de filas. Vivienda recreativa se reorganiza con `set_index().unstack()`, operaciones presentes en el parcial, sin agregación.

In [4]:
bases_trimestrales = []
control_cruces = []
procedencia = []
for trimestre in carpetas:
    base = tablas[trimestre, "CARACTERISTICAS_GENERALES"].copy()
    if trimestre == 1:
        procedencia.extend({
            "columna": c, "modulo": "CARACTERISTICAS_GENERALES", "variable_original": c
        } for c in base.columns)
    for modulo in modulos[1:]:
        fuente = tablas[trimestre, modulo].copy()
        if modulo == "VIVIENDA_USO_RECREATIVO":
            clave = ["llave_hogar"]
            fuente["presente"] = "1"
            derecha = fuente.set_index(["llave_hogar", "Orden"]).unstack("Orden")
            nombres = {
                par: modulo + "__" + par[0] + "__registro_" + str(par[1])
                for par in derecha.columns
            }
            procedencia.extend({
                "columna": nombre, "modulo": modulo, "variable_original": par[0]
            } for par, nombre in nombres.items())
            derecha.columns = [nombres[par] for par in derecha.columns]
            derecha = derecha.reset_index()
        else:
            clave = llaves[modulo]
            nombres = {c: modulo + "__" + c for c in fuente.columns if c not in clave}
            procedencia.extend({
                "columna": nombre, "modulo": modulo, "variable_original": original
            } for original, nombre in nombres.items())
            derecha = fuente.rename(columns=nombres)
        cobertura = derecha[clave].merge(
            base[clave].drop_duplicates(), on=clave, how="left",
            validate="one_to_one", indicator=True
        )
        huerfanas = int((cobertura["_merge"] == "left_only").sum())
        assert huerfanas == 0, f"Registros sin enlace: T{trimestre} {modulo}"
        indicador = "presente__" + modulo
        derecha[indicador] = True
        filas_antes = len(base)
        base = base.merge(derecha, on=clave, how="left", validate="many_to_one")
        base[indicador] = base[indicador].eq(True)
        assert len(base) == filas_antes
        control_cruces.append({
            "trimestre": trimestre, "modulo": modulo,
            "filas_antes": filas_antes, "filas_despues": len(base),
            "personas_sin_registro_modulo": int((~base[indicador]).sum()),
            "registros_huerfanos": huerfanas,
        })
    base["trimestre"] = trimestre
    base["anio"] = 2024
    bases_trimestrales.append(base)
df_consolidado = pd.concat(bases_trimestrales, ignore_index=True)
control_cruces = pd.DataFrame(control_cruces)
procedencia = pd.DataFrame(procedencia).drop_duplicates().reset_index(drop=True)
llave_final = ["trimestre"] + llave_persona
assert not df_consolidado.duplicated(llave_final).any()
assert len(df_consolidado) == inventario.loc[
    inventario["modulo"] == "CARACTERISTICAS_GENERALES", "filas"
].sum()
display(control_cruces)
display(df_consolidado.groupby("trimestre").size().rename("personas").to_frame())
print("Dimensión consolidada:", df_consolidado.shape)

,trimestre,modulo,filas_antes,filas_despues,personas_sin_registro_modulo,registros_huerfanos
0,1,EDUCACION,31185,31185,894,0
1,1,FUERZA_DE_TRABAJO,31185,31185,3784,0
2,1,COMPLEMENTARIAS,31185,31185,3784,0
3,1,TURISMO,31185,31185,3784,0
4,1,EXCURSIONISMO,31185,31185,3784,0
5,1,HOGAR,31185,31185,0,0
6,1,VIVIENDA,31185,31185,0,0
7,1,VIVIENDA_USO_RECREATIVO,31185,31185,30767,0
8,2,EDUCACION,30542,30542,886,0
9,2,FUERZA_DE_TRABAJO,30542,30542,3673,0


,personas
trimestre,
1,31185
2,30542
3,30740
4,29512


Dimensión consolidada: (121979, 278)


### 2.2 Diagnóstico de la base consolidada
Dos personas con las mismas respuestas no son necesariamente duplicados. Los controles conservan identificadores; ninguna alerta elimina columnas o filas automáticamente.

In [5]:
display(pd.DataFrame({
    "control": ["Duplicados exactos", "Duplicados persona-trimestre"],
    "cantidad": [df_consolidado.duplicated().sum(),
                 df_consolidado.duplicated(llave_final).sum()],
}))
calidad_consolidada = pd.DataFrame({
    "tipo": df_consolidado.dtypes.astype(str),
    "faltantes": df_consolidado.isna().sum(),
    "porcentaje_faltante": df_consolidado.isna().mean() * 100,
    "valores_distintos": df_consolidado.nunique(),
})
display(calidad_consolidada.sort_values("porcentaje_faltante", ascending=False).head(30))
display(calidad_consolidada.loc[calidad_consolidada["valores_distintos"] <= 1])

,control,cantidad
0,Duplicados exactos,0
1,Duplicados persona-trimestre,0


,tipo,faltantes,porcentaje_faltante,valores_distintos
EXCURSIONISMO__P7587S8A3,string,121979,100.000000,0
VIVIENDA_USO_RECREATIVO__P38S3A1__registro_3,string,121979,100.000000,0
TURISMO__P7581S11A2,string,121979,100.000000,0
EXCURSIONISMO__P7587S4A3,string,121979,100.000000,0
EXCURSIONISMO__P7587S7A3,string,121978,99.999180,1
EXCURSIONISMO__P7587S2A3,string,121973,99.995081,6
TURISMO__P7581S8A3,string,121972,99.994261,4
TURISMO__P7581S5A3,string,121971,99.993441,5
VIVIENDA_USO_RECREATIVO__P38S2A1__registro_3,string,121970,99.992622,2
EXCURSIONISMO__P7587S5A3,string,121969,99.991802,7


,tipo,faltantes,porcentaje_faltante,valores_distintos
TURISMO__P7581S11A2,string,121979,100.000000,0
EXCURSIONISMO__P7587S4A3,string,121979,100.000000,0
EXCURSIONISMO__P7587S7A3,string,121978,99.999180,1
EXCURSIONISMO__P7587S8A3,string,121979,100.000000,0
HOGAR__Secuencia_p,string,0,0.000000,1
presente__HOGAR,bool,0,0.000000,1
VIVIENDA__Secuencia_p,string,0,0.000000,1
VIVIENDA__Secuencia_encuesta,string,0,0.000000,1
VIVIENDA__Orden,string,0,0.000000,1
presente__VIVIENDA,bool,0,0.000000,1


In [6]:
import re

TECNICAS_REDUNDANTES = [
    c for c in df_consolidado.columns
    if re.search(r"__(Secuencia_encuesta|Fex_c|llave_hogar|Directorio|"
                 r"Secuencia_p|Orden)(__registro_\d+)?$", c)
    or c.startswith("presente__") or "__presente__" in c
]

RENOMBRAR = {
    "Fex_c": "fex_trim",
    "VIVIENDA__area": "ciudad",
    "P6020": "sexo",
    "P6040": "edad",
    "P6050": "parentesco_jefe",
    "EDUCACION__P6170": "asiste_estudio",
    "EDUCACION__P6210": "nivel_educativo",
    "FUERZA_DE_TRABAJO__P6240": "actividad_principal",
    "HOGAR__P36": "hogar_viv_recreativa",
    "HOGAR__P545": "fuente_ingreso_principal",
    "HOGAR__P547": "ingreso_mensual_hogar",
    "TURISMO__P7570": "viajo_pernoctando",
    "TURISMO__P7571S1": "viaje_dentro_pais",
    "TURISMO__P7571S2": "viaje_fuera_pais",
    "TURISMO__P7573S1": "destino_depto",
    "TURISMO__P7573S2": "destino_municipio",
    "TURISMO__P7579": "motivo_no_viajar",
    "TURISMO__P7580": "motivo_viaje",
    "TURISMO__P7580S1": "viaje_con_quien",
    "TURISMO__P549": "fecha_inicio_viaje",
    "TURISMO__P549S1": "fecha_fin_viaje",
    "TURISMO__P15001": "reserva_por_app",
    "TURISMO__P7575": "transporte_principal",
    "TURISMO__P7576": "pago_paquete",
    "TURISMO__P7576S2": "paquete_n_personas",
    "TURISMO__P7576S3": "paquete_monto",
    "TURISMO__P7578": "gasto_total_viaje",
}
# alojamiento: S{n} = tipo usado, S{n}A1 = noches. El S7 usa A2, no A1.
for s in [1, 2, 3, 4, 5, 8, 10]:
    RENOMBRAR[f"TURISMO__P7574S{s}"] = f"aloj{s:02d}_uso"
    RENOMBRAR[f"TURISMO__P7574S{s}A1"] = f"aloj{s:02d}_noches"
RENOMBRAR["TURISMO__P7574S7"] = "aloj07_uso"
RENOMBRAR["TURISMO__P7574S7A2"] = "aloj07_noches"

# rubros: S{n} = ¿gastó?, A1 = monto, A2 = nº personas, A3 = submonto.
# El S11 no tiene A3.
for r in [1, 3, 4, 5, 6, 7, 8, 9, 10, 11]:
    RENOMBRAR[f"TURISMO__P7581S{r}"] = f"rubro{r:02d}_gasto_si"
    RENOMBRAR[f"TURISMO__P7581S{r}A1"] = f"rubro{r:02d}_monto"
    RENOMBRAR[f"TURISMO__P7581S{r}A2"] = f"rubro{r:02d}_n_personas"
    if r != 11:
        RENOMBRAR[f"TURISMO__P7581S{r}A3"] = f"rubro{r:02d}_monto_alt"

assert set(RENOMBRAR) <= set(df_consolidado.columns)
df_consolidado = (df_consolidado
                  .drop(columns=TECNICAS_REDUNDANTES)
                  .rename(columns=RENOMBRAR))
procedencia["columna"] = procedencia["columna"].replace(RENOMBRAR)

print("Técnicas eliminadas:", len(TECNICAS_REDUNDANTES))
print("Renombradas:", len(RENOMBRAR), "| Dimensión:", df_consolidado.shape)

Técnicas eliminadas: 47
Renombradas: 82 | Dimensión: (121979, 231)


In [7]:
def diagnostico(df, columnas, aplica):
    sub = df.loc[aplica, columnas]
    return pd.DataFrame({
        "n_aplica": len(sub),
        "no_nulos": sub.notna().sum(),
        "pct_faltante": (100 * sub.isna().mean()).round(2),
        "n_valores": sub.nunique(dropna=True),
    }).reset_index(names="columna").sort_values("pct_faltante")

es_viajero = df_consolidado["gasto_total_viaje"].notna()
cols_tur = [c for c in df_consolidado.columns
            if c.startswith(("rubro", "aloj"))
            or c in ("gasto_total_viaje", "transporte_principal", "motivo_viaje",
                     "pago_paquete", "fecha_inicio_viaje", "destino_depto")]

print("Viajeros con detalle:", int(es_viajero.sum()))
display(diagnostico(df_consolidado, cols_tur, es_viajero))

Viajeros con detalle: 6762


,columna,n_aplica,no_nulos,pct_faltante,n_valores
0,motivo_viaje,6762,6762,0.00,9
1,destino_depto,6762,6762,0.00,32
2,fecha_inicio_viaje,6762,6762,0.00,383
3,aloj01_uso,6762,6762,0.00,2
5,aloj05_uso,6762,6762,0.00,2
...,...,...,...,...,...
49,rubro06_monto_alt,6762,25,99.63,10
37,rubro03_monto_alt,6762,24,99.65,13
45,rubro05_monto_alt,6762,8,99.88,5
57,rubro08_monto_alt,6762,7,99.90,4


### 2.3 Selección, corrección de tipos y eliminación de filas
Completa las listas tras interpretar las tablas. `columnas_numericas` debe contener cantidades, no códigos categóricos. No reemplaces globalmente 9/99/999: su significado depende de la pregunta. Las conversiones se detienen si encuentran valores no convertibles.

In [8]:
RUBROS = [1, 3, 4, 5, 6, 7, 8, 9, 10]   # el 11 se descarta: 27 montos en todo 2024
ALOJ = [1, 2, 3, 4, 5, 7, 8, 10]

columnas_descartar = (
    [c for c in df_consolidado.columns
     if c.startswith(("EXCURSIONISMO__", "COMPLEMENTARIAS__",
                      "VIVIENDA_USO_RECREATIVO__"))]
    + [c for c in df_consolidado.columns if c.endswith("_monto_alt")]
    + [c for c in df_consolidado.columns if c.startswith("rubro11_")]
    + ["motivo_no_viajar", "destino_municipio", "TURISMO__P7573S3",
       "TURISMO__P7571S1A1", "TURISMO__P7571S1A2", "TURISMO__P7571S1A3",
       "TURISMO__P7571S2A1", "TURISMO__P7580S3"]
)
columnas_numericas = (
    ["edad", "ingreso_mensual_hogar", "gasto_total_viaje",
     "paquete_monto", "paquete_n_personas", "fex_trim"]
    + [f"rubro{r:02d}_monto" for r in RUBROS]
    + [f"rubro{r:02d}_n_personas" for r in RUBROS]
    + [f"aloj{a:02d}_noches" for a in ALOJ]
)
columnas_categoricas = [
    "ciudad", "sexo", "nivel_educativo", "actividad_principal",
    "hogar_viv_recreativa", "fuente_ingreso_principal", "motivo_viaje",
    "viaje_con_quien", "transporte_principal", "pago_paquete",
    "reserva_por_app", "destino_depto",
] + [f"rubro{r:02d}_gasto_si" for r in RUBROS] + [f"aloj{a:02d}_uso" for a in ALOJ]

reemplazos = {}
columnas_exigir_dato = ["gasto_total_viaje"]

assert not set(columnas_descartar) & set(llave_final)
assert not set(columnas_numericas) & set(columnas_categoricas)
assert set(columnas_descartar + columnas_numericas + columnas_categoricas
           + columnas_exigir_dato + list(reemplazos)) <= set(df_consolidado.columns)
df_trabajo = df_consolidado.drop(columns=columnas_descartar).copy()
df_trabajo = df_trabajo.replace(reemplazos)
conversiones = []
for columna in columnas_numericas:
    convertida = pd.to_numeric(df_trabajo[columna], errors="coerce")
    invalidos = df_trabajo[columna].notna() & convertida.isna()
    conversiones.append({"columna": columna, "no_convertibles": int(invalidos.sum())})
    assert not invalidos.any(), f"Revisa los valores no convertibles de {columna}."
    df_trabajo[columna] = convertida.astype(float)
for columna in columnas_categoricas:
    df_trabajo[columna] = df_trabajo[columna].astype("category")
if columnas_exigir_dato:
    df_trabajo = df_trabajo.dropna(subset=columnas_exigir_dato)
display(pd.DataFrame(conversiones, columns=["columna", "no_convertibles"]))
print("Filas retiradas:", len(df_consolidado) - len(df_trabajo))

,columna,no_convertibles
0,edad,0
1,ingreso_mensual_hogar,0
2,gasto_total_viaje,0
3,paquete_monto,0
4,paquete_n_personas,0
5,fex_trim,0
6,rubro01_monto,0
7,rubro03_monto,0
8,rubro04_monto,0
9,rubro05_monto,0


Filas retiradas: 115217


### 2.4 Tratamiento de faltantes
Se reutilizan `fillna()` y `mean()`. El estudiante debe definir una máscara booleana `aplica` por regla, según el formulario. La media se calcula solo sobre las filas aplicables. No se rellenan automáticamente saltos del cuestionario.

Formato: `{"columna": nombre, "metodo": "media" o "valor", "aplica": mascara, "valor": valor_elegido}`. La máscara debe conservar el índice de `df_trabajo`.

In [9]:
todos = pd.Series(True, index=df_trabajo.index)

reglas_imputacion = (
    [{"columna": f"rubro{r:02d}_monto", "aplica": todos,
      "metodo": "valor", "valor": 0.0} for r in RUBROS]
    + [{"columna": f"aloj{a:02d}_noches", "aplica": todos,
        "metodo": "valor", "valor": 0.0} for a in ALOJ]
    + [{"columna": "paquete_monto", "aplica": todos,
        "metodo": "valor", "valor": 0.0}]
)
registro_imputacion = []
for regla in reglas_imputacion:
    columna = regla["columna"]
    aplica = regla["aplica"]
    assert isinstance(aplica, pd.Series) and aplica.index.equals(df_trabajo.index)
    assert pd.api.types.is_bool_dtype(aplica) and not aplica.isna().any()
    antes = int(df_trabajo.loc[aplica, columna].isna().sum())
    if regla["metodo"] == "media":
        assert columna in columnas_numericas
        valor = df_trabajo.loc[aplica, columna].mean()
    else:
        assert regla["metodo"] == "valor"
        valor = regla["valor"]
    assert pd.notna(valor), "Valor de imputación no válido."
    df_trabajo.loc[aplica, columna] = df_trabajo.loc[aplica, columna].fillna(valor)
    despues = int(df_trabajo.loc[aplica, columna].isna().sum())
    registro_imputacion.append({
        "columna": columna, "metodo": regla["metodo"], "valor": valor,
        "imputados": antes - despues, "faltantes_aplicables": despues,
    })
registro_imputacion = pd.DataFrame(registro_imputacion, columns=[
    "columna", "metodo", "valor", "imputados", "faltantes_aplicables"
])
display(registro_imputacion)

,columna,metodo,valor,imputados,faltantes_aplicables
0,rubro01_monto,valor,0.0,5383,0
1,rubro03_monto,valor,0.0,5491,0
2,rubro04_monto,valor,0.0,2621,0
3,rubro05_monto,valor,0.0,5891,0
4,rubro06_monto,valor,0.0,5949,0
5,rubro07_monto,valor,0.0,5754,0
6,rubro08_monto,valor,0.0,6222,0
7,rubro09_monto,valor,0.0,6030,0
8,rubro10_monto,valor,0.0,2987,0
9,aloj01_noches,valor,0.0,6589,0


In [14]:
df_trabajo["duracion_dias"] = (
    pd.to_datetime(df_trabajo["fecha_fin_viaje"], errors="coerce")
    - pd.to_datetime(df_trabajo["fecha_inicio_viaje"], errors="coerce")
).dt.days
df_trabajo["log_gasto_total"] = np.log1p(df_trabajo["gasto_total_viaje"])

COLS_RUBRO = [f"rubro{r:02d}_monto" for r in RUBROS]
df_trabajo["suma_rubros"] = df_trabajo[COLS_RUBRO].sum(axis=1)
df_trabajo["coherencia"] = df_trabajo["suma_rubros"] / df_trabajo["gasto_total_viaje"]

print("Sin desglose (suma_rubros == 0):", int((df_trabajo["suma_rubros"] == 0).sum()))
display(df_trabajo.loc[df_trabajo["suma_rubros"] > 0, "coherencia"]
        .describe(percentiles=[.05, .25, .5, .75, .95]).round(3))

# los que no tienen desglose no sirven para composición
df_trabajo = df_trabajo.loc[df_trabajo["suma_rubros"] > 0].copy()

for r in RUBROS:
    df_trabajo[f"share_rubro{r:02d}"] = (
        df_trabajo[f"rubro{r:02d}_monto"] / df_trabajo["suma_rubros"])

# para que el Punto 3 analice las variables que realmente se van a usar
columnas_numericas += (["duracion_dias", "log_gasto_total", "suma_rubros"]
                       + [f"share_rubro{r:02d}" for r in RUBROS])

print("Base para clustering:", df_trabajo.shape)

Sin desglose (suma_rubros == 0): 2033


count    4729.000
mean        0.953
std         0.272
min         0.000
5%          0.378
25%         1.000
50%         1.000
75%         1.000
95%         1.000
max         5.000
Name: coherencia, dtype: float64

Base para clustering: (4729, 135)


### 2.5 Interpretación del estudiante
Pendiente: variables descartadas y motivo; faltantes frente a no aplica; correcciones; filas excluidas; método y alcance de cada imputación.

## Punto 3: Análisis descriptivo y tratamiento de atípicos
### 3.1 Estadísticos y frecuencias
Las salidas se activan al completar las listas de variables.

In [10]:
if columnas_numericas:
    display(df_trabajo[columnas_numericas].describe().T)
else:
    print("Pendiente: seleccionar variables cuantitativas.")
if columnas_categoricas:
    display(pd.concat({
        c: df_trabajo[c].value_counts(dropna=False) for c in columnas_categoricas
    }, names=["variable", "categoria"]).rename("frecuencia").to_frame())

,count,mean,std,min,25%,50%,75%,max
edad,6762.0,4.050237e+01,1.798756e+01,10.000000,2.600000e+01,3.900000e+01,5.400000e+01,9.300000e+01
ingreso_mensual_hogar,6752.0,3.981446e+06,3.793638e+06,98.000000,1.600000e+06,3.000000e+06,5.000000e+06,3.500000e+07
gasto_total_viaje,6762.0,9.827778e+05,1.539955e+06,98.000000,2.000000e+05,5.000000e+05,1.100000e+06,2.500000e+07
paquete_monto,6762.0,8.535378e+04,5.565467e+05,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,1.040000e+07
paquete_n_personas,281.0,2.416370e+00,2.079049e+00,1.000000,1.000000e+00,2.000000e+00,3.000000e+00,3.000000e+01
fex_trim,6762.0,9.909766e+02,1.888260e+03,16.657892,1.630321e+02,3.115026e+02,6.865485e+02,1.940226e+04
rubro01_monto,6762.0,1.023955e+05,4.755994e+05,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,1.200000e+07
rubro03_monto,6762.0,1.854746e+04,6.317868e+04,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,1.600000e+06
rubro04_monto,6762.0,1.970789e+05,3.876168e+05,0.000000,0.000000e+00,7.000000e+04,2.500000e+05,9.000000e+06
rubro05_monto,6762.0,2.360866e+04,1.393102e+05,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,3.000000e+06


frecuencia
variable   categoria            
ciudad     15                799
           17                537
           11                435
           68                389
           19                372
...                          ...
aloj07_uso 1                  76
aloj08_uso 2                6535
           1                 227
aloj10_uso 2                6663
           1                  99

[149 rows x 1 columns]

### 3.2 Distribuciones y correlación
El estudiante selecciona la variable que desea graficar.

In [11]:
variable_grafico = None
if variable_grafico is not None:
    assert variable_grafico in columnas_numericas
    sns.histplot(data=df_trabajo, x=variable_grafico)
    plt.show()
    sns.boxplot(data=df_trabajo, x=variable_grafico)
    plt.show()
if len(columnas_numericas) > 1:
    display(df_trabajo[columnas_numericas].corr())

,edad,ingreso_mensual_hogar,gasto_total_viaje,paquete_monto,paquete_n_personas,fex_trim,rubro01_monto,rubro03_monto,rubro04_monto,rubro05_monto,...,rubro09_n_personas,rubro10_n_personas,aloj01_noches,aloj02_noches,aloj03_noches,aloj04_noches,aloj05_noches,aloj07_noches,aloj08_noches,aloj10_noches
edad,1.000000,0.053687,0.037812,-0.010985,-0.102833,-0.011419,0.032285,0.086985,0.129284,0.036557,...,-0.038368,0.007231,0.087733,-0.018119,0.009426,-0.003345,0.042379,0.025989,0.005585,0.011353
ingreso_mensual_hogar,0.053687,1.000000,0.322931,0.074312,0.236468,0.126537,0.233574,0.063694,0.213371,0.188333,...,0.235262,0.178257,0.136359,0.129786,-0.009573,-0.010809,-0.104772,-0.029713,0.010505,0.011301
gasto_total_viaje,0.037812,0.322931,1.000000,0.344760,0.182022,-0.016107,0.634368,0.262347,0.571673,0.474664,...,0.623043,0.313185,0.038543,0.409335,-0.007186,0.067021,-0.037121,-0.018073,0.107809,0.050368
paquete_monto,-0.010985,0.074312,0.344760,1.000000,0.166674,0.023019,-0.024302,-0.008271,-0.051685,0.016549,...,0.120680,0.029002,-0.017531,0.183397,-0.003513,0.088613,-0.075133,-0.000037,-0.015753,-0.009765
paquete_n_personas,-0.102833,0.236468,0.182022,0.166674,1.000000,-0.026809,-0.016969,-0.017248,-0.014062,0.035645,...,1.000000,0.989481,NaN,0.037250,-0.053060,-0.021230,-0.070985,0.017000,0.039313,-0.023859
fex_trim,-0.011419,0.126537,-0.016107,0.023019,-0.026809,1.000000,0.022022,-0.050694,0.051466,-0.020212,...,0.083870,0.056451,0.051503,0.015630,-0.008311,0.017599,-0.018003,-0.013867,0.009279,-0.008507
rubro01_monto,0.032285,0.233574,0.634368,-0.024302,-0.016969,0.022022,1.000000,0.233983,0.528868,0.569914,...,0.433072,0.250798,-0.024611,0.374757,-0.004054,0.036050,-0.104116,-0.012658,0.072739,0.018062
rubro03_monto,0.086985,0.063694,0.262347,-0.008271,-0.017248,-0.050694,0.233983,1.000000,0.259299,0.217605,...,0.211558,-0.044175,-0.002115,0.149768,0.043340,-0.000630,0.058613,0.015521,0.035644,-0.016983
rubro04_monto,0.129284,0.213371,0.571673,-0.051685,-0.014062,0.051466,0.528868,0.259299,1.000000,0.388863,...,0.455541,0.313922,0.110976,0.200783,-0.002706,0.007712,0.088122,-0.012870,0.092958,0.045684
rubro05_monto,0.036557,0.188333,0.474664,0.016549,0.035645,-0.020212,0.569914,0.217605,0.388863,1.000000,...,0.323028,0.132140,0.053878,0.168168,0.010878,0.006090,0.011114,0.002336,0.000662,-0.008319


### 3.3 Candidatos atípicos mediante IQR
Función original del profesor. Se aplica solo a las cantidades seleccionadas. Una marca no justifica automáticamente borrar o imputar el valor.

In [12]:
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return data[(data[column] < lower_bound) | (data[column] > upper_bound)]

resumen_atipicos = []
for columna in columnas_numericas:
    candidatos = detect_outliers_iqr(df_trabajo, columna)
    resumen_atipicos.append({
        "variable": columna, "observados": int(df_trabajo[columna].notna().sum()),
        "candidatos_iqr": len(candidatos),
    })
resumen_atipicos = pd.DataFrame(resumen_atipicos, columns=[
    "variable", "observados", "candidatos_iqr"
])
display(resumen_atipicos)
if variable_grafico is not None:
    display(detect_outliers_iqr(df_trabajo, variable_grafico)[
        llave_final + [variable_grafico]
    ].head(20))

,variable,observados,candidatos_iqr
0,edad,6762,0
1,ingreso_mensual_hogar,6752,332
2,gasto_total_viaje,6762,673
3,paquete_monto,6762,281
4,paquete_n_personas,281,1
5,fex_trim,6762,1070
6,rubro01_monto,6762,1379
7,rubro03_monto,6762,1271
8,rubro04_monto,6762,441
9,rubro05_monto,6762,871


### 3.4 Tratamiento e interpretación del estudiante
Pendiente: revisión y justificación. Si decides retirar registros, indica sus índices. No se excluyen candidatos IQR automáticamente.

In [13]:
indices_excluir = []
assert set(indices_excluir) <= set(df_trabajo.index)
df_preparado = df_trabajo.drop(index=indices_excluir).copy()
assert not df_preparado.duplicated(llave_final).any()
display(pd.DataFrame({
    "etapa": ["Consolidada", "Después de faltantes", "Después de revisión de atípicos"],
    "filas": [len(df_consolidado), len(df_trabajo), len(df_preparado)],
    "columnas": [df_consolidado.shape[1], df_trabajo.shape[1], df_preparado.shape[1]],
}))

,etapa,filas,columnas
0,Consolidada,121979,231
1,Después de faltantes,6762,122
2,Después de revisión de atípicos,6762,122


## Punto 4: Preprocesamiento para el modelado — punto de parada
### 4.1 Selección y verificación previa
La matriz requiere decisiones sobre población, variables, categorías, faltantes y escala. Los identificadores y factores se conservan para trazabilidad; no son características por defecto.

La reducción de dimensionalidad queda pendiente según las variables. No se ejecutan PCA, codificación, escalado ni k-means.

In [15]:
variables_modelado = (
    [f"share_rubro{r:02d}" for r in RUBROS]
    + ["log_gasto_total", "duracion_dias"]
)
decisiones_revisadas = True
X_previo = df_preparado.loc[:, variables_modelado].copy()
display(pd.DataFrame({
    "tipo": X_previo.dtypes.astype(str),
    "faltantes": X_previo.isna().sum(),
    "valores_distintos": X_previo.nunique(),
}))
print("Decisiones revisadas:", decisiones_revisadas)
print("Variables seleccionadas:", len(variables_modelado))
print("Estado: base consolidada disponible; matriz para clustering pendiente.")

KeyError: "None of [Index(['share_rubro01', 'share_rubro03', 'share_rubro04', 'share_rubro05',\n       'share_rubro06', 'share_rubro07', 'share_rubro08', 'share_rubro09',\n       'share_rubro10', 'log_gasto_total', 'duracion_dias'],\n      dtype='str')] are in the [columns]"

### 4.2 Exportación y comprobación
`egit_2024_consolidada.csv.gz` reúne los cuatro trimestres. `egit_2024_preparacion_actual.csv.gz` guarda las decisiones configuradas, **no certifica que esté listo para k-means**. Son CSV comprimidos con gzip.

Se guardan los controles completos y la procedencia de las columnas.

In [ ]:
ruta_consolidada = SALIDA / "egit_2024_consolidada.csv.gz"
df_consolidado.to_csv(ruta_consolidada, index=False, compression="gzip")
df_preparado.to_csv(SALIDA / "egit_2024_preparacion_actual.csv.gz",
                   index=False, compression="gzip")
for nombre, reporte in {
    "inventario": inventario, "calidad_origen": calidad_origen, "esquemas": esquemas,
    "control_llaves": control_llaves, "control_cruces": control_cruces,
    "procedencia_columnas": procedencia, "imputaciones": registro_imputacion,
    "atipicos": resumen_atipicos,
}.items():
    reporte.to_csv(SALIDA / (nombre + ".csv"), index=False)
calidad_consolidada.to_csv(SALIDA / "calidad_consolidada.csv", index_label="columna")
recarga = pd.read_csv(ruta_consolidada, dtype="string",
                     keep_default_na=False, na_values=[""])
assert recarga.shape == df_consolidado.shape
assert recarga.equals(df_consolidado.astype("string"))
print("Exportación verificada:", ruta_consolidada.name, recarga.shape)

Exportación verificada: egit_2024_consolidada.csv.gz (121979, 278)


### 4.3 Pendientes para continuar
Interpretar las tablas, elegir población y variables, consultar categorías y saltos, configurar limpieza e imputación y revisar atípicos. Luego decidir representación y escala, reducción de dimensionalidad si corresponde y finalmente k-means.

Se revisó el [tutorial de Real Python solicitado](https://realpython.com/k-means-clustering-python/); queda como referencia para la siguiente etapa, sin incorporar su código. La consulta de módulos usa el [diccionario DANE EGIT 2024](https://microdatos.dane.gov.co/index.php/catalog/864/data-dictionary) y [vivienda recreativa](https://microdatos.dane.gov.co/index.php/catalog/864/data-dictionary/F3?file_name=Vivienda+uso+recreativo).

Apoyo de IA: programación y comprobaciones técnicas. Interpretación, decisiones metodológicas y conclusiones: pendientes del estudiante.